In [11]:
import sys
import torch
import transformers
import vllm
import bitsandbytes as bnb

print("Python executable:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)
print("transformers:", transformers.__version__)
print("vllm imported")
print("bitsandbytes imported")

Python executable: /home/folin/CSE 151B/151B_SP26_Competition/.venv/bin/python
CUDA available: True
CUDA version: 12.1
Torch version: 2.5.1+cu121
transformers: 5.7.0
vllm imported
bitsandbytes imported


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [12]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "1"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

In [13]:
import time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

from tqdm.auto import tqdm

In [14]:
DATA_PATH = "data/public.jsonl"

data = [json.loads(line) for line in open(DATA_PATH, "r", encoding="utf-8")]

print(f"Loaded {len(data)} examples")
print(f"MCQ: {sum(bool(x.get('options')) for x in data)}")
print(f"Free-form: {sum(not bool(x.get('options')) for x in data)}")

Loaded 1126 examples
MCQ: 375
Free-form: 751


In [15]:
def item_to_text(item):
    """
    Convert one problem into a plain text input for classical ML models.
    This intentionally destroys sequence/reasoning structure, which is the point
    of the BoW baseline.
    """
    question = item["question"]

    if item.get("options"):
        labels = [chr(65 + i) for i in range(len(item["options"]))]
        options_text = "\n".join(
            f"{label}. {option}" for label, option in zip(labels, item["options"])
        )
        return question + "\n\nOptions:\n" + options_text

    return question


def answer_to_label(item):
    """
    Convert the ground-truth answer into a classification label.
    
    MCQ:
        "C"
    Free-form single:
        "105950"
    Free-form multiple:
        "580, 660, 80"
    """
    ans = item["answer"]

    if item.get("options"):
        return str(ans).strip().upper()

    if isinstance(ans, list):
        return ", ".join(str(x).strip() for x in ans)

    return str(ans).strip()


def label_to_response(label):
    """
    Convert a predicted label into the response format expected by the scorer.
    """
    return f"\\boxed{{{label}}}"


records = []
for idx, item in enumerate(data):
    records.append({
        "idx": idx,
        "id": item["id"],
        "is_mcq": bool(item.get("options")),
        "text": item_to_text(item),
        "label": answer_to_label(item),
        "item": item,
    })

df = pd.DataFrame(records)

print(df.head())
print("\nLabel examples:")
print(df[["id", "is_mcq", "label"]].head(10))

   idx  id  is_mcq                                               text  \
0    0   0   False  Find the sum of the first $325$ positive even ...   
1    1   1    True  $int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ...   
2    2   2   False  A roasted turkey is taken from an oven when it...   
3    3   3   False       Reduce the fraction ${\frac{25}{40}}$. [ANS]   
4    4   4    True  Given $u(x, y) = x^3 + 6x^2y - 3xy^2 - 2y^3$, ...   

                                label  \
0                         325*(1+325)   
1                                   F   
2  143.224229233795, 2.32624773420025   
3                                 5/8   
4                                   C   

                                                item  
0  {'question': 'Find the sum of the first $325$ ...  
1  {'question': '$int_{-infty}^{+infty} frac{a^{3...  
2  {'question': 'A roasted turkey is taken from a...  
3  {'question': 'Reduce the fraction ${\frac{25}{...  
4  {'question': 'Given $u(x, y) = x^3 +

In [16]:
RANDOM_SEED = 42

train_df, dev_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=df["is_mcq"],  # keeps MCQ/free-form ratio similar
)

train_df = train_df[train_df["is_mcq"] == True]
dev_df = dev_df[dev_df["is_mcq"] == True]

print("Train size:", len(train_df))
print("Dev size:", len(dev_df))

print("\nTrain format counts:")
print(train_df["is_mcq"].value_counts())

print("\nDev format counts:")
print(dev_df["is_mcq"].value_counts())

Train size: 300
Dev size: 75

Train format counts:
is_mcq
True    300
Name: count, dtype: int64

Dev format counts:
is_mcq
True    75
Name: count, dtype: int64


In [17]:
def train_logistic_bow(train_df, dev_df, max_features=10000):
    start_time = time.time()

    # 🔴 MCQ ONLY
    train_df = train_df[train_df["is_mcq"] == True]
    dev_df = dev_df[dev_df["is_mcq"] == True]

    vectorizer = CountVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=(1, 1),   # 🔴 remove bigrams
        max_features=max_features,
    )

    X_train = vectorizer.fit_transform(train_df["text"])
    X_dev = vectorizer.transform(dev_df["text"])

    y_train = train_df["label"].values
    y_dev = dev_df["label"].values

    clf = LogisticRegression(
        max_iter=500,
        solver="saga",
    )

    print("Training BoW + Logistic Regression...")
    clf.fit(X_train, y_train)

    pred_labels = clf.predict(X_dev)

    elapsed = time.time() - start_time

    print(f"Done in {elapsed:.2f} seconds")
    print("Dev accuracy:", accuracy_score(y_dev, pred_labels))

    return clf, vectorizer

bow_lr_run = train_logistic_bow(train_df, dev_df)

Training BoW + Logistic Regression...
Done in 0.27 seconds
Dev accuracy: 0.12


/home/folin/CSE 151B/151B_SP26_Competition/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [22]:
def clean_label(x):
    if isinstance(x, list):
        return ", ".join(str(v).strip() for v in x)
    return str(x).strip()

train_df = train_df.copy()
dev_df = dev_df.copy()

train_df["label"] = train_df["label"].apply(clean_label)
dev_df["label"] = dev_df["label"].apply(clean_label)

train_df = train_df[train_df["is_mcq"] == True].copy()
dev_df = dev_df[dev_df["is_mcq"] == True].copy()

train_df["label"] = train_df["label"].apply(clean_label)
dev_df["label"] = dev_df["label"].apply(clean_label)

print(train_df["label"].unique())

<StringArray>
['C', 'H', 'F', 'I', 'D', 'B', 'J', 'E', 'G', 'A']
Length: 10, dtype: str


In [23]:
def train_mlp_bow(train_df, dev_df, max_features=5000):
    start_time = time.time()

    # MCQ only is much safer for MLP baseline
    train_df = train_df[train_df["is_mcq"] == True].copy()
    dev_df = dev_df[dev_df["is_mcq"] == True].copy()

    # Force clean string labels
    train_df["label"] = train_df["label"].astype(str).str.strip()
    dev_df["label"] = dev_df["label"].astype(str).str.strip()

    vectorizer = CountVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=(1, 1),
        max_features=max_features,
    )

    X_train = vectorizer.fit_transform(train_df["text"]).astype(np.float32).toarray()
    X_dev = vectorizer.transform(dev_df["text"]).astype(np.float32).toarray()

    y_train = train_df["label"].values
    y_dev = dev_df["label"].values

    print("Unique labels:", sorted(set(y_train)))
    print("Training BoW + MLP...")

    clf = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        batch_size=32,
        max_iter=50,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=RANDOM_SEED,
        verbose=True,
    )

    clf.fit(X_train, y_train)

    pred_labels = clf.predict(X_dev)

    elapsed = time.time() - start_time

    result_df = dev_df.copy()
    result_df["pred_label"] = pred_labels
    result_df["response"] = result_df["pred_label"].apply(label_to_response)
    result_df["exact_label_correct"] = result_df["pred_label"] == result_df["label"]

    print(f"Done in {elapsed:.2f} seconds")
    print("Dev exact-label accuracy:", accuracy_score(y_dev, pred_labels))

    return {
        "name": "bow_mlp_mcq_only",
        "vectorizer": vectorizer,
        "model": clf,
        "result_df": result_df,
        "elapsed_sec": elapsed,
    }

bow_mlp_run = train_mlp_bow(train_df, dev_df)

Unique labels: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
Training BoW + MLP...
Iteration 1, loss = 2.52992011


TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
import sys
sys.path.insert(0, ".")

from judger import Judger

judger = Judger(strict_extract=False)


def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == str(gold_letter).strip().upper()


def score_response(item, response):
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        return score_mcq(response, str(gold))

    gold_list = gold if isinstance(gold, list) else [gold]

    try:
        return judger.auto_judge(
            pred=response,
            gold=gold_list,
            options=[[]] * len(gold_list),
        )
    except Exception:
        return False


def evaluate_baseline_run(run):
    result_df = run["result_df"].copy()

    scored_rows = []
    for _, row in tqdm(result_df.iterrows(), total=len(result_df), desc=f"Scoring {run['name']}"):
        item = row["item"]
        response = row["response"]
        correct = score_response(item, response)

        scored_rows.append({
            "id": row["id"],
            "is_mcq": row["is_mcq"],
            "gold": row["label"],
            "pred_label": row["pred_label"],
            "response": response,
            "correct": correct,
        })

    scored_df = pd.DataFrame(scored_rows)

    mcq_df = scored_df[scored_df["is_mcq"] == True]
    free_df = scored_df[scored_df["is_mcq"] == False]

    def safe_acc(x):
        return 100 * x["correct"].mean() if len(x) else 0.0

    summary = {
        "run_name": run["name"],
        "num_dev": len(scored_df),
        "num_mcq": len(mcq_df),
        "num_free": len(free_df),
        "mcq_acc": safe_acc(mcq_df),
        "free_form_acc": safe_acc(free_df),
        "overall_acc": safe_acc(scored_df),
        "train_time_sec": run["elapsed_sec"],
    }

    print("=" * 60)
    print(run["name"])
    print("=" * 60)
    print(f"MCQ       : {mcq_df['correct'].sum():4d} / {len(mcq_df):4d} ({summary['mcq_acc']:.2f}%)")
    print(f"Free-form : {free_df['correct'].sum():4d} / {len(free_df):4d} ({summary['free_form_acc']:.2f}%)")
    print(f"Overall   : {scored_df['correct'].sum():4d} / {len(scored_df):4d} ({summary['overall_acc']:.2f}%)")
    print("=" * 60)

    return scored_df, summary


bow_lr_scored, bow_lr_summary = evaluate_baseline_run(bow_lr_run)
bow_mlp_scored, bow_mlp_summary = evaluate_baseline_run(bow_mlp_run)

summary_df = pd.DataFrame([bow_lr_summary, bow_mlp_summary])
summary_df

In [ ]:
RESULTS_DIR = Path("results/classical_baselines")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

bow_lr_scored.to_json(
    RESULTS_DIR / "bow_logistic_regression_dev_results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

bow_mlp_scored.to_json(
    RESULTS_DIR / "bow_mlp_dev_results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

summary_df.to_csv(RESULTS_DIR / "classical_baseline_summary.csv", index=False)

print("Saved results to:", RESULTS_DIR)

In [ ]:
def show_failures(scored_df, n=10):
    wrong = scored_df[scored_df["correct"] == False].head(n)

    for _, row in wrong.iterrows():
        print("=" * 100)
        print("ID:", row["id"])
        print("Type:", "MCQ" if row["is_mcq"] else "Free-form")
        print("Gold:", row["gold"])
        print("Pred:", row["pred_label"])
        print("Response:", row["response"])


print("BoW Logistic Regression failures:")
show_failures(bow_lr_scored, n=5)

print("\n\nBoW MLP failures:")
show_failures(bow_mlp_scored, n=5)

In [ ]:
mcq_df = df[df["is_mcq"] == True].copy()

mcq_train_df, mcq_dev_df = train_test_split(
    mcq_df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=mcq_df["label"],
)

print("MCQ train:", len(mcq_train_df))
print("MCQ dev:", len(mcq_dev_df))
print(mcq_train_df["label"].value_counts())

In [ ]:
mcq_bow_lr_run = train_logistic_bow(mcq_train_df, mcq_dev_df)
mcq_bow_lr_scored, mcq_bow_lr_summary = evaluate_baseline_run(mcq_bow_lr_run)

In [ ]:
mcq_bow_mlp_run = train_mlp_bow(mcq_train_df, mcq_dev_df, max_features=5000)
mcq_bow_mlp_scored, mcq_bow_mlp_summary = evaluate_baseline_run(mcq_bow_mlp_run)

In [ ]:
mcq_summary_df = pd.DataFrame([mcq_bow_lr_summary, mcq_bow_mlp_summary])
mcq_summary_df.to_csv(RESULTS_DIR / "mcq_only_classical_baseline_summary.csv", index=False)
mcq_summary_df

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [12]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")


Scoring:   0%|          | 5/1126 [00:00<00:33, 33.44it/s]

Scoring complete. 5 results.


## 8. Summary

Print accuracy broken down by question type.

In [13]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    2 /    3  (66.67%)
  Overall    :    3 /    5  (60.00%)
